# orbit-agent: agent v4 decision head

Trains the small classifier on top of a frozen image model (`src/model/head.py`) on a GPU.
Code: https://github.com/devxsubh/orbit-agent.git (branch `main`). Photos: the Kaggle copy of AID (jiayuanchengala/aid-scene-classification-datasets).
Results are saved in `results/` (Output tab).

In [ ]:
# 1. Get the project code (into /tmp, so it isn't copied into the output)
import glob, os, shutil, subprocess

def run(command):
    """Run a shell command and STOP the notebook if it fails."""
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)

run("rm -rf /tmp/orbit-agent && git clone --depth 1 -b main https://github.com/devxsubh/orbit-agent.git /tmp/orbit-agent")
os.chdir("/tmp/orbit-agent")
run("git log --oneline -1")
run("rm -f experiments/head_aid_*.csv")  # so only results made in THIS run are kept

In [ ]:
# 2. Which GPU did we get? Where are the AID photos?
run("nvidia-smi --query-gpu=name,memory.total --format=csv")
found = glob.glob("/kaggle/input/**/AID/Airport", recursive=True)
assert found, "AID dataset not attached (add it in the notebook's Input panel)"
os.environ["AID_DIR"] = os.path.dirname(found[0])
print("AID photos in", os.environ["AID_DIR"])

In [ ]:
# 3. Make embeddings with the frozen model, train the head
run("python -u -m src.model.head --backbone siglip")

In [ ]:
# 4. Keep the results (Kaggle saves everything in /kaggle/working)
os.makedirs("/kaggle/working/results", exist_ok=True)
files = glob.glob("experiments/head_aid_*.csv") + glob.glob("data/processed/embeddings_aid_*.npz")
assert files, "No results were made"
for f in files:
    shutil.copy(f, "/kaggle/working/results/")
run("ls -lh /kaggle/working/results")